# 2.9 · 最大似然估计 / Maximum Likelihood Estimation

> **课程定位**
> 0.9 推过 Bernoulli/Normal 的解析 MLE；本课补齐**完整工具链**：数值优化求 MLE、**Fisher 信息给标准误**、渐近正态给 CI、似然比检验。**几乎所有"模型训练"（逻辑回归、GLM、神经网络的交叉熵）本质都是 MLE**——这一课是它们共同的统计学说明书。
> The full MLE toolchain: numerical optimization, Fisher information for standard errors, asymptotic normality, likelihood ratio tests. Nearly all "model training" is MLE under the hood.

> 💡 **面试相关**
> - "MLE 的性质（一致性/渐近正态/不变性）" ★★★★
> - "Fisher 信息是什么" ★★★（量化岗/统计岗）
> - "为什么最小化交叉熵 = MLE" ★★★★★（ML 岗必考，0.11 讲过，这里补严格版）
> - "MLE 什么时候会坏" ★★★

---

## 目录
1. [似然：把数据当常数，把参数当变量 ⭐](#1)
2. [解析解回顾 + 指数分布推导](#2)
3. [数值 MLE：scipy.optimize 通用配方 ⭐](#3)
4. [Fisher 信息与标准误 ⭐](#4)
5. [MLE 三大渐近性质（模拟验证）](#5)
6. [似然比检验 LRT](#6)
7. [⭐ ML 联结：交叉熵 / MSE 都是 MLE](#7)
8. [⚠ MLE 失灵的场景](#8)
9. [实战：censored 数据的生存参数估计](#9)
10. [小结](#10)


<a id="1"></a>
## 1. 似然：把数据当常数，把参数当变量 ⭐ / Likelihood

同一个函数 $p(x \mid \theta)$，两种读法：

| 读法 | 固定谁 | 变谁 | 名字 |
|---|---|---|---|
| 概率 | $\theta$ | $x$ | $p(x \mid \theta)$，对 $x$ 积分 = 1 |
| **似然** | **$x$（数据已观测）** | **$\theta$** | $\mathcal{L}(\theta) = \prod_i p(x_i \mid \theta)$，对 $\theta$ 积分**不必** = 1 |

$$\hat{\theta}_{\mathrm{MLE}} = \arg\max_\theta \;\ell(\theta), \qquad \ell(\theta) = \sum_i \log p(x_i \mid \theta)$$

**为什么取 log**：积变和（数值不下溢 + 求导容易），且 log 单调不改最大值位置。


In [ ]:
import numpy as np
import scipy.stats as st
import scipy.optimize as opt
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
rng = np.random.default_rng(42)

# 可视化: 同一枚硬币 10 次 7 正, 似然函数长什么样
# Coin: 7 heads in 10 flips — the likelihood as a function of p
k, n = 7, 10
ps = np.linspace(0.01, 0.99, 300)
lik = ps**k * (1-ps)**(n-k)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.plot(ps, lik, lw=2)
ax.axvline(k/n, color="r", ls="--", label=f"MLE = k/n = {k/n}")
ax.set_xlabel("p (参数)"); ax.set_ylabel("L(p)")
ax.set_title("Likelihood of p given 7 heads / 10 flips")
ax.legend(); plt.tight_layout(); plt.show()


<a id="2"></a>
## 2. 解析解回顾 + 指数分布推导 / Analytic MLEs

| 分布 | MLE | 备注 |
|---|---|---|
| Bernoulli($p$) | $\hat{p} = k/n$ | 0.9 推过 |
| Normal($\mu, \sigma^2$) | $\hat\mu = \bar{x}$, $\hat\sigma^2 = \frac{1}{n}\sum(x_i-\bar x)^2$ | 方差有偏（÷n）|
| Poisson($\lambda$) | $\hat\lambda = \bar{x}$ | 2.2 用过 |
| **Exponential($\lambda$)** | 推导如下 ↓ | |

**指数分布推导**（密度 $\lambda e^{-\lambda x}$）：
$$\ell(\lambda) = n\log\lambda - \lambda \sum_i x_i
\;\Rightarrow\; \frac{d\ell}{d\lambda} = \frac{n}{\lambda} - \sum_i x_i = 0
\;\Rightarrow\; \boxed{\hat\lambda = \frac{1}{\bar{x}}}$$

直觉：率 = 1/平均等待时间。二阶导 $-n/\lambda^2 < 0$ 确认是最大值。


<a id="3"></a>
## 3. 数值 MLE：通用配方 ⭐ / Numerical MLE

绝大多数模型**没有解析解**（Gamma、Weibull、逻辑回归、混合模型）。通用配方：

```
1. 写出负对数似然 nll(θ)        ← 负号: 优化器都是 minimize
2. scipy.optimize.minimize(nll, θ0, method="L-BFGS-B", bounds=...)
3. Hessian 逆 → 标准误 (下一节)
```

用 **Gamma 分布**演示（shape $\alpha$ 无解析解）：


In [ ]:
# 真参数 / Truth
true_alpha, true_beta = 3.0, 1.5            # shape, rate
x = rng.gamma(true_alpha, 1/true_beta, 500)

def nll_gamma(params, data):
    a, b = params                            # shape, rate
    if a <= 0 or b <= 0: return np.inf
    # logpdf 手写: a·ln b - ln Γ(a) + (a-1)ln x - b x
    from scipy.special import gammaln
    return -np.sum(a*np.log(b) - gammaln(a) + (a-1)*np.log(data) - b*data)

res = opt.minimize(nll_gamma, x0=[1.0, 1.0], args=(x,),
                   method="L-BFGS-B", bounds=[(1e-6, None)]*2)
a_hat, b_hat = res.x
print(f"真值:  α={true_alpha}, β={true_beta}")
print(f"MLE :  α={a_hat:.3f}, β={b_hat:.3f}")
print(f"scipy 内置 fit 对照: {st.gamma.fit(x, floc=0)}  (shape, loc, scale=1/β={1/b_hat:.3f})")


**手写 NLL + L-BFGS-B 与 scipy 内置 `fit` 一致**。这套配方对任何能写出密度的模型通用——0.10 节的优化器在这里找到统计学的"客户"。
This recipe works for any model with a writable density — the optimizers from 0.10 meet their statistical customer.


<a id="4"></a>
## 4. Fisher 信息与标准误 ⭐ / Fisher Information & SEs

**Fisher 信息** = 对数似然的曲率期望：
$$I(\theta) = -\mathbb{E}\Big[\frac{\partial^2 \ell}{\partial \theta^2}\Big]$$

**直觉**：似然峰越**尖**（曲率大）→ 数据对 $\theta$ 越"挑剔" → 信息越多 → 估计越准。

**渐近方差**（单参数）：
$$\mathrm{Var}(\hat\theta) \approx \frac{1}{n\,I_1(\theta)} \qquad \Rightarrow \qquad \mathrm{SE}(\hat\theta) = \sqrt{[\,\mathbf{H}^{-1}\,]_{jj}}$$

实操：**优化收敛点的 Hessian 逆的对角线开根 = 各参数的标准误**。这正是 statsmodels/R 回归输出里 `std err` 列的来源。
In practice: invert the Hessian at the optimum; the square roots of its diagonal are the standard errors — exactly the `std err` column in any regression output.


In [ ]:
# 数值 Hessian → SE → 95% CI / Numerical Hessian to CIs
def numerical_hessian(f, x0, eps=1e-5):
    k = len(x0); H = np.zeros((k, k))
    for i in range(k):
        for j in range(k):
            e_i, e_j = np.zeros(k), np.zeros(k)
            e_i[i] = e_j[j] = eps
            H[i, j] = (f(x0+e_i+e_j) - f(x0+e_i-e_j) - f(x0-e_i+e_j) + f(x0-e_i-e_j)) / (4*eps**2)
    return H

H = numerical_hessian(lambda p: nll_gamma(p, x), res.x)
cov = np.linalg.inv(H)                      # NLL 的 Hessian 逆 = 渐近协方差
se = np.sqrt(np.diag(cov))

print(f"{'param':<7} {'MLE':>8} {'SE':>8} {'95% CI':>20} {'真值':>6}")
for name, est, s, true in [("alpha", a_hat, se[0], true_alpha), ("beta", b_hat, se[1], true_beta)]:
    lo, hi = est - 1.96*s, est + 1.96*s
    inside = "✓" if lo <= true <= hi else "✗"
    print(f"{name:<7} {est:>8.3f} {se[0] if name=='alpha' else se[1]:>8.3f} "
          f"[{lo:.3f}, {hi:.3f}]{'':>3} {true:>6} {inside}")


<a id="5"></a>
## 5. MLE 三大渐近性质（模拟验证）/ The Three Asymptotic Properties

| 性质 | 表述 |
|---|---|
| **一致性** / Consistency | $\hat\theta \xrightarrow{p} \theta^\star$ — n 够大必收敛到真值 |
| **渐近正态** / Asymptotic normality | $\sqrt{n}(\hat\theta - \theta^\star) \xrightarrow{d} \mathcal{N}(0, I_1^{-1})$ |
| **渐近有效** / Efficiency | 方差达到 Cramér-Rao 下界——**正则条件下没有更准的无偏估计** |

外加一条好用的**不变性** / invariance：$g(\hat\theta)_{\mathrm{MLE}} = g(\hat\theta_{\mathrm{MLE}})$——估计 $\lambda$ 的 MLE 后想要 $1/\lambda$ 的 MLE？直接取倒数。


In [ ]:
# 模拟验证一致性 + 渐近正态 (指数分布 λ̂ = 1/x̄)
true_lam = 2.0
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))

# 一致性: 不同 n 下估计的分布收紧到真值
for n_ in [20, 100, 500]:
    lam_hats = 1 / rng.exponential(1/true_lam, (4000, n_)).mean(axis=1)
    axes[0].hist(lam_hats, bins=60, density=True, alpha=0.55, label=f"n={n_}")
axes[0].axvline(true_lam, color="k", ls="--"); axes[0].legend()
axes[0].set_title("Consistency: λ̂ concentrates on truth")

# 渐近正态: √n(λ̂-λ) vs 理论 N(0, λ²)  [I₁(λ)=1/λ² → 方差 λ²]
n_ = 500
lam_hats = 1 / rng.exponential(1/true_lam, (20_000, n_)).mean(axis=1)
z = np.sqrt(n_) * (lam_hats - true_lam)
xs = np.linspace(-8, 8, 200)
axes[1].hist(z, bins=80, density=True, alpha=0.7)
axes[1].plot(xs, st.norm.pdf(xs, 0, true_lam), "r-", lw=2, label=f"N(0, λ²={true_lam**2})")
axes[1].legend(); axes[1].set_title("Asymptotic normality: √n(λ̂−λ)")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. 似然比检验 LRT / Likelihood Ratio Test

比较**嵌套**模型（简单 $H_0$ ⊂ 复杂 $H_1$）：

$$\Lambda = 2\,\big[\ell(\hat\theta_{H_1}) - \ell(\hat\theta_{H_0})\big] \;\sim\; \chi^2_{\Delta \mathrm{df}} \quad (\text{Wilks 定理})$$

**直觉**：复杂模型的似然必然 ≥ 简单模型（参数多）——LRT 问的是"**多出来的似然超过随机预期了吗**"。这是 GLM/逻辑回归里比较模型的标准武器，也是 AIC（2.2 节）背后的理论亲戚。
The richer model always fits better; LRT asks whether the gain beats chance. Wilks' theorem gives the chi-square yardstick.


In [ ]:
# LRT: 这批数据用 Exponential 够吗, 还是需要 Gamma?
# (Exponential = Gamma 的 α=1 特例 → 嵌套)
x_test = rng.gamma(2.5, 1.0, 300)            # 真相: Gamma(2.5) — 指数不够

ll_gamma = -nll_gamma(st.gamma.fit(x_test, floc=0)[0::2], x_test)   # (shape, rate=1/scale)
# 修正: gamma.fit 返回 (shape, loc, scale); rate = 1/scale
p_g = st.gamma.fit(x_test, floc=0)
ll_g = np.sum(st.gamma.logpdf(x_test, p_g[0], scale=p_g[2]))
lam_h = 1 / x_test.mean()
ll_e = np.sum(st.expon.logpdf(x_test, scale=1/lam_h))

LR = 2 * (ll_g - ll_e)
p_val = st.chi2.sf(LR, df=1)                 # 多 1 个参数 (shape)
print(f"logL(Gamma) = {ll_g:.1f},  logL(Expon) = {ll_e:.1f}")
print(f"LRT 统计量 = {LR:.1f},  p = {p_val:.2e}")
print(f"→ {'拒绝 Exponential, 需要 Gamma 的 shape 参数' if p_val < 0.05 else '指数分布够用'}")


<a id="7"></a>
## 7. ⭐ ML 联结：交叉熵 / MSE 都是 MLE

**(1) 分类 → 交叉熵**。模型输出 $q_\theta(y\mid x)$，对 i.i.d. 数据：
$$\hat\theta = \arg\max \sum_i \log q_\theta(y_i \mid x_i) = \arg\min \underbrace{-\tfrac{1}{n}\sum_i \log q_\theta(y_i \mid x_i)}_{\text{交叉熵损失}}$$
**最小化交叉熵 ≡ 类别分布的 MLE**（0.11 的信息论版，这里是统计版）。

**(2) 回归 + 高斯噪声 → MSE**。设 $y = f_\theta(x) + \varepsilon, \;\varepsilon \sim \mathcal{N}(0, \sigma^2)$：
$$-\ell(\theta) = \frac{1}{2\sigma^2}\sum_i (y_i - f_\theta(x_i))^2 + \text{const} \;\propto\; \mathrm{MSE}$$
**最小化 MSE ≡ 高斯噪声假设下的 MLE**。

**(3) 推论**：换噪声假设 = 换损失。Laplace 噪声 → MAE（L1）；重尾噪声 → Huber。**损失函数不是拍脑袋，是噪声模型的化身**——这是 Part 4 回归损失选择的统一视角。
Losses aren't arbitrary — each is the MLE of a noise model. Swap the noise, swap the loss.


<a id="8"></a>
## 8. ⚠ MLE 失灵的场景 / When MLE Breaks

| 场景 | 症状 | 解法 |
|---|---|---|
| **小样本** | 偏差明显（如 $\hat\sigma^2$ 低估）| 修正（÷(n-1)）/ 贝叶斯（2.10）|
| **完美分离**（逻辑回归）| $\|\hat\beta\| \to \infty$，不收敛 | 正则化 = MAP（2.10）|
| **混合模型** | 似然无界（一个分量方差→0 套住单点）| 约束方差下限 / 贝叶斯 |
| **参数在边界** | Wilks 定理失效（LRT 的 χ² 不对）| boundary-corrected 检验 |
| **模型错настройка** | 收敛到"伪真值"（KL 最近的错模型）| 模型诊断（2.2 的 QQ/AIC）|

> 💡 其中"完美分离 → 正则化救场"在 Part 5 逻辑回归会真实遇到。


<a id="9"></a>
## 9. 实战：censored 数据的生存参数估计 / Hands-on: Censored Data

**MLE 的独门绝技**：优雅处理**删失数据**——20.1 生存分析的统计核心，这里先尝鲜。

**场景**：测灯泡寿命，实验 1000 小时结束时还有灯泡活着（右删失）。直接删掉删失样本 = 严重低估寿命（幸存者偏差的镜像！）。

**MLE 解法**：删失样本贡献**生存概率**而非密度：
$$\ell(\lambda) = \underbrace{\sum_{\text{died}} \log f(t_i)}_{\text{死亡: 密度}} + \underbrace{\sum_{\text{alive}} \log S(c)}_{\text{删失: }\Pr(T > c)}$$


In [ ]:
# 真寿命 Exponential(λ=1/800), 实验在 1000 小时截止
true_scale = 800.0
lifetime = rng.exponential(true_scale, 300)
CENSOR = 1000.0
observed = np.minimum(lifetime, CENSOR)
died = lifetime <= CENSOR
print(f"删失比例: {(~died).mean():.0%} 的灯泡在实验结束时还活着")

# 方法 1 (错): 只用死亡样本的均值 / Wrong: drop the censored
naive = observed[died].mean()

# 方法 2 (对): censored MLE
def nll_censored(log_scale):
    s = np.exp(log_scale)
    ll_death = st.expon.logpdf(observed[died], scale=s).sum()
    ll_alive = st.expon.logsf(CENSOR, scale=s) * (~died).sum()
    return -(ll_death + ll_alive)

res = opt.minimize_scalar(nll_censored, bounds=(np.log(100), np.log(5000)), method="bounded")
mle_scale = np.exp(res.x)

print(f"\n真平均寿命       = {true_scale:.0f} 小时")
print(f"只用死亡样本     = {naive:.0f}   ← 低估 {(1-naive/true_scale)*100:.0f}%! (删失版幸存者偏差)")
print(f"censored MLE     = {mle_scale:.0f}   ← 准确恢复 ✓")


**只用死亡样本低估 ~30%**（活得久的全被截断在视野外）；censored MLE 把"还活着"本身当证据用，**准确恢复真寿命**。

同款数学应用于：用户流失时间（多数用户还没流失）、贷款违约（多数贷款未到期）、设备维修周期——**业务数据里删失无处不在**，这是 MLE 框架最被低估的实战价值。
Churn, default, repair cycles — censoring is everywhere in business data. Handling it natively is MLE's most underrated superpower.


<a id="10"></a>
## 10. 小结 / Summary

```
MLE 工具链:
  写 NLL → minimize (L-BFGS-B) → Hessian⁻¹ → SE → CI
  性质: 一致 + 渐近正态 (√n, I₁⁻¹) + 有效 (Cramér-Rao) + 不变性
  检验: LRT = 2Δℓ ~ χ²(Δdf)  (Wilks)

ML 統一视角 ⭐:
  交叉熵 = 类别 MLE | MSE = 高斯 MLE | MAE = Laplace MLE
  → 损失函数 = 噪声模型的化身

失灵: 小样本偏差 / 完美分离 / 无界似然 → 正则化·贝叶斯救场 (2.10)
绝技: censored 数据 — 删失样本贡献 log S(c)
```

### 💡 面试速查
1. **似然 vs 概率**：固定数据变参数 vs 固定参数变数据
2. **SE 从哪来**：NLL 的 Hessian 逆对角线开根（回归表里的 std err）
3. **交叉熵 = MLE** 两行推导要会写
4. **MLE 方差有偏**（÷n），2.1 的 n-1 在这里接上
5. **删失数据**：删掉 = 幸存者偏差；MLE 用 $\log S(c)$ 原生处理

### 下一节
**2.10 贝叶斯估计**——MLE 的"对手戏"：先验 × 似然 = 后验，共轭、可信区间、以及"正则化其实是 MAP"。
